## Loading camera-trap-vehicle-classifier weights into PyTorch and compiling to Torchscript

Things to note: 
- the model was trained on a GPU so we need to load weights and re-compile to CPU
- it's important to check what version of torchvision (if used here) and torch you're running in this notebook environment & be sure they match the versions pinned in the deployment container's Dockerfile

In [1]:
import torch
import pytorch_lightning as pl
from typing import Dict, List, Optional
import timm

/opt/homebrew/Caskroom/miniforge/base/envs/camera-trap-vehicle-classifier/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class VehicleClassifier(pl.LightningModule):
    """
    Lightning module for vehicle classification
    """

    def __init__(
        self,
        model_name: str,
        num_classes: int,
        class_to_idx: Dict[str, int],
        learning_rate: float = 1e-4,
        freeze_layers: bool = True,
        output_dir: str = None
    ):
        super().__init__()
        print("Initializing VehicleClassifier...")
        self.model_name = model_name
        self.num_classes = num_classes
        self.class_to_idx = class_to_idx
        self.learning_rate = learning_rate
        self.freeze_layers = freeze_layers
        self.output_dir = output_dir

        # Save hyperparameters for checkpoint loading (exclude output_dir as it's training-specific)
        self.save_hyperparameters(ignore=['output_dir'])

        # Create idx_to_class mapping for convenience
        self.idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

        # Initialize preprocessing parameters (will be set by data module)
        self.image_size = None
        self.normalization_mean = None
        self.normalization_std = None

        # Load model
        self.set_up_model()

        # For tracking metrics
        self.validation_step_outputs = []

        # Create metrics CSV file and save class names
        if self.output_dir:
            self.metrics_file = os.path.join(self.output_dir, 'training_metrics.csv')

            # Save class names to text file
            self._save_class_names()

            # Save detailed class mapping as JSON
            self._save_class_mapping()

    def set_up_metrics_file(self, resume_from=None):
        """
        Setup metrics CSV file, handling resume case
        """

        if not self.output_dir:
            return

        # Check whether we're resuming
        if resume_from and os.path.exists(self.metrics_file):
            print(f"Resuming training - will append to existing metrics file: {self.metrics_file}")
        else:
            # Initialize new CSV file
            print(f"Creating new metrics file: {self.metrics_file}")
            with open(self.metrics_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['epoch', 'train_loss', 'val_loss', 'val_accuracy', 'val_macro_accuracy'])

    def _save_class_names(self):
        """
        Save class names to classes.txt file in output directory
        """

        if self.output_dir:
            classes_file = os.path.join(self.output_dir, 'classes.txt')
            with open(classes_file, 'w') as f:
                # Write class names in order of their indices
                for i in range(self.num_classes):
                    f.write(f"{self.idx_to_class[i]}\n")
            print(f"Saved class names to: {classes_file}")

    def _save_class_mapping(self):
        """
        Save detailed class mapping and metadata as JSON
        """

        if self.output_dir:
            metadata = {
                'model_name': self.model_name,
                'num_classes': self.num_classes,
                'class_to_idx': self.class_to_idx,
                'idx_to_class': self.idx_to_class,
                'class_names': [self.idx_to_class[i] for i in range(self.num_classes)],
                'learning_rate': self.learning_rate,
                'freeze_layers': self.freeze_layers,
                'preprocessing': {
                    'image_size': getattr(self, 'image_size', None),
                    'normalization_mean': getattr(self, 'normalization_mean', None),
                    'normalization_std': getattr(self, 'normalization_std', None)
                },
                'training_info': {
                    'framework': 'pytorch_lightning',
                    'model_type': 'fine_tuned_classifier',
                    'metadata_format': 'coco_json'
                }
            }

            metadata_file = os.path.join(self.output_dir, 'model_metadata.json')
            with open(metadata_file, 'w') as f:
                json.dump(metadata, f, indent=2)
            print(f"Saved model metadata to: {metadata_file}")

    def on_train_start(self):
        """
        Called when training starts, saves preprocessing parameters
        """

        # Get preprocessing parameters from data module
        if hasattr(self.trainer, 'datamodule'):
            dm = self.trainer.datamodule
            if hasattr(dm, 'image_size'):
                self.image_size = dm.image_size
                self.normalization_mean = dm.normalization_mean
                self.normalization_std = dm.normalization_std

                # Update hyperparameters to include preprocessing info
                if hasattr(self, 'hparams'):
                    self.hparams.update({
                        'image_size': self.image_size,
                        'normalization_mean': self.normalization_mean,
                        'normalization_std': self.normalization_std
                    })

                print(f"Saved preprocessing parameters:")
                print(f"  Image size: {self.image_size}")
                print(f"  Mean: {self.normalization_mean}")
                print(f"  Std: {self.normalization_std}")

                # Re-save metadata with preprocessing info
                self._save_class_mapping()

    def set_up_model(self):
        """
        Set up the model with appropriate freezing
        """
        
        print(f"Setting up model: {self.model_name}")
        is_timm_model = self.model_name.startswith('timm/')

        if is_timm_model:
            print("Using timm model architecture")
            # Use timm model
            timm_model_name = self.model_name[5:]  # Remove 'timm/' prefix
            self.model = timm.create_model(timm_model_name, pretrained=True, num_classes=self.num_classes)

            if self.freeze_layers:
                self._freeze_timm_layers()

        else:
            print("Using Hugging Face model architecture")
            # Use Hugging Face model
            self.model = AutoModelForImageClassification.from_pretrained(
                self.model_name,
                num_labels=self.num_classes,
                ignore_mismatched_sizes=True
            )

            if self.freeze_layers:
                self._freeze_hf_layers()

        # ...is this a timm model or a Hugging Face model?

    def _freeze_timm_layers(self):
        """
        Freeze layers for timm models (keep classifier head + last 2-4 blocks unfrozen)
        """

        print("Freezing timm model layers...")

        # Freeze all parameters first
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze classifier head
        if hasattr(self.model, 'head'):
            for param in self.model.head.parameters():
                param.requires_grad = True

        # For EVA02 models, unfreeze last few blocks
        if hasattr(self.model, 'blocks'):
            num_blocks = len(self.model.blocks)
            blocks_to_unfreeze = 3  # Unfreeze last 3 blocks

            for i in range(max(0, num_blocks - blocks_to_unfreeze), num_blocks):
                for param in self.model.blocks[i].parameters():
                    param.requires_grad = True

            print(f"Unfroze classifier head and last {blocks_to_unfreeze} blocks out of {num_blocks}")

        # Count trainable parameters
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")

    def _freeze_hf_layers(self):
        """
        Freeze layers for Hugging Face models
        """

        print("Freezing Hugging Face model layers...")

        # Freeze all parameters first
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze classifier
        if hasattr(self.model, 'classifier'):
            for param in self.model.classifier.parameters():
                param.requires_grad = True

        # For ViT models, unfreeze last few encoder layers
        if hasattr(self.model, 'vit') and hasattr(self.model.vit, 'encoder'):
            layers = self.model.vit.encoder.layer
            layers_to_unfreeze = 3  # Unfreeze last 3 layers

            for i in range(max(0, len(layers) - layers_to_unfreeze), len(layers)):
                for param in layers[i].parameters():
                    param.requires_grad = True

            print(f"Unfroze classifier and last {layers_to_unfreeze} encoder layers out of {len(layers)}")

        # Count trainable parameters
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")

    def forward(self, x):
        if self.model_name.startswith('timm/'):
            return self.model(x)
        else:
            return self.model(x).logits

    def training_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = F.cross_entropy(logits, labels)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = F.cross_entropy(logits, labels)
        preds = torch.argmax(logits, dim=1)

        self.validation_step_outputs.append({
            'val_loss': loss,
            'preds': preds,
            'labels': labels
        })

        return loss

    def on_validation_epoch_end(self):

        # Compute metrics
        all_preds = torch.cat([x['preds'] for x in self.validation_step_outputs])
        all_labels = torch.cat([x['labels'] for x in self.validation_step_outputs])
        avg_loss = torch.stack([x['val_loss'] for x in self.validation_step_outputs]).mean()

        # Overall accuracy
        accuracy = accuracy_score(all_labels.cpu(), all_preds.cpu())

        # Macro accuracy (per-class accuracy averaged)
        class_accuracies = []
        for class_idx in range(self.num_classes):
            mask = all_labels == class_idx
            if mask.sum() > 0:
                class_acc = (all_preds[mask] == all_labels[mask]).float().mean()
                class_accuracies.append(class_acc.item())

        macro_accuracy = sum(class_accuracies) / len(class_accuracies) if class_accuracies else 0.0

        # Log metrics
        self.log('val_loss', avg_loss, prog_bar=True)
        self.log('val_accuracy', accuracy, prog_bar=True)
        self.log('val_macro_accuracy', macro_accuracy, prog_bar=True)

        # Write to CSV
        if self.output_dir:
            with open(self.metrics_file, 'a', newline='') as f:
                writer = csv.writer(f)
                train_loss = self.trainer.callback_metrics.get('train_loss', 0.0)
                writer.writerow([
                    self.current_epoch,
                    float(train_loss),
                    float(avg_loss),
                    float(accuracy),
                    float(macro_accuracy)
                ])

        # Clear outputs
        self.validation_step_outputs.clear()

    def configure_optimizers(self):

        # Use different learning rates for different parts
        if self.freeze_layers:
            # Higher learning rate for unfrozen layers
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                lr=self.learning_rate,
                weight_decay=0.01
            )
        else:
            # Lower learning rate for full fine-tuning
            optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.learning_rate / 10,
                weight_decay=0.01
            )

        # Cosine annealing scheduler
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.trainer.max_epochs,
            eta_min=self.learning_rate / 100
        )

        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch'
            }
        }

# ...class VehicleClassifier

def load_model_from_checkpoint(checkpoint_path, class_names=None):
    """
    Load model from checkpoint, extracting necessary parameters.

    Args:
        checkpoint_path: Path to the checkpoint file
        class_names: List of class names (if not provided, will use generic names)

    Returns:
        tuple: (model, class_to_idx, idx_to_class, model_name)
    """

    print(f"Loading checkpoint: {checkpoint_path}")

    # Load checkpoint to extract metadata
    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        print("Checkpoint loaded successfully")
    except Exception as e:
        raise Exception(f"Failed to load checkpoint: {e}")

    # Extract hyperparameters
    if 'hyper_parameters' not in checkpoint:
        raise Exception("No hyperparameters found in checkpoint. This checkpoint may be from an older training script.")

    hparams = checkpoint['hyper_parameters']
    print(f"Found hyperparameters: {list(hparams.keys())}")

    # Extract required parameters
    model_name = hparams.get('model_name', 'unknown_model')
    num_classes = hparams.get('num_classes')
    saved_class_to_idx = hparams.get('class_to_idx', {})

    if num_classes is None:
        raise Exception("num_classes not found in checkpoint hyperparameters")

    print(f"Model: {model_name}")
    print(f"Number of classes: {num_classes}")

    # Create class mappings
    if class_names:
        if len(class_names) != num_classes:
            raise Exception(f"Provided class names ({len(class_names)}) don't match model classes ({num_classes})")

        # Use provided class names
        class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
        idx_to_class = {idx: cls for idx, cls in enumerate(class_names)}
        print(f"Using provided class names: {class_names[:5]}{'...' if len(class_names) > 5 else ''}")

    elif saved_class_to_idx:
        # Use class mapping from checkpoint
        class_to_idx = saved_class_to_idx
        idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}
        print(f"Using class names from checkpoint: {list(class_to_idx.keys())[:5]}{'...' if len(class_to_idx) > 5 else ''}")

    else:
        # Fallback to generic class names
        class_to_idx = {f"class_{i}": i for i in range(num_classes)}
        idx_to_class = {i: f"class_{i}" for i in range(num_classes)}
        print(f"Using generic class names: class_0, class_1, ..., class_{num_classes-1}")

    # Load the model using Lightning's load_from_checkpoint
    try:
        model = VehicleClassifier.load_from_checkpoint(
            checkpoint_path,
            model_name=model_name,
            num_classes=num_classes,
            class_to_idx=class_to_idx
        )
        print("Model loaded successfully")
        return model, class_to_idx, idx_to_class, model_name

    except Exception as e:
        raise Exception(f"Failed to load model from checkpoint: {e}")


def load_class_names(class_file):
    """
    Load class names from file
    """

    try:
        with open(class_file, 'r') as f:
            class_names = [line.strip() for line in f.readlines() if line.strip()]
        print(f"Loaded {len(class_names)} class names from {class_file}")
        return class_names
    except Exception as e:
        raise Exception(f"Failed to load class names from {class_file}: {e}")

In [19]:
from torchvision import transforms

def get_transforms_from_checkpoint(checkpoint_path, input_size=None):
    """
    Get the appropriate transforms for inference from a checkpoint.

    Args:
        checkpoint_path (str): Path to the trained model checkpoint
        input_size (tuple): Override input size (height, width) if specified

    Returns:
        transform: Transform pipeline for inference
    """

    try:
        # Load checkpoint to get preprocessing info
        checkpoint = torch.load(checkpoint_path, map_location='cpu')

        if 'hyper_parameters' not in checkpoint:
            raise Exception("No hyperparameters found in checkpoint")

        hparams = checkpoint['hyper_parameters']

        # Extract preprocessing parameters (these should be present in new checkpoints)
        if input_size is not None:
            size = input_size
            print(f"Using provided input size: {size}")
        else:
            size = hparams.get('image_size')
            if size is None:
                raise Exception("image_size not found in checkpoint hyperparameters. This " + \
                                "model may have been trained with an older script.")
            print(f"Using image size from checkpoint: {size}")

        mean = hparams.get('normalization_mean')
        std = hparams.get('normalization_std')

        if mean is None or std is None:
            raise Exception("normalization_mean or normalization_std not found in checkpoint " + \
                            "hyperparameters. This model may have been trained with an older script.")

        print(f"Using normalization from checkpoint: mean {mean}, std {std}")

    except Exception as e:
        raise Exception(f"Failed to extract preprocessing parameters from checkpoint: {e}")

    # Create transform pipeline
    transform_list = [
        transforms.Resize(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ]

    transform = transforms.Compose(transform_list)
    print(f"Created transform pipeline: {transform}")

    return transform

In [3]:
# load class names
class_file = "./original-model/classes.txt"
class_names = load_class_names(class_file)

Loaded 4 class names from ./original-model/classes.txt


In [4]:
class_names

['car_or_truck', 'motorbike', 'mountain_biker', 'quad']

In [5]:
# Load model from checkpoint
checkpoint_path = "./original-model/camera-trap-vehicle-classifier.2025.07.09.ckpt"
device = torch.device('cpu')
model, class_to_idx, idx_to_class, model_name = load_model_from_checkpoint(checkpoint_path, class_names)
model = model.to(device)

Loading checkpoint: ./original-model/camera-trap-vehicle-classifier.2025.07.09.ckpt
Checkpoint loaded successfully
Found hyperparameters: ['model_name', 'num_classes', 'class_to_idx', 'learning_rate', 'freeze_layers', 'image_size', 'normalization_mean', 'normalization_std']
Model: timm/eva02_large_patch14_448.mim_m38m_ft_in22k_in1k
Number of classes: 4
Using provided class names: ['car_or_truck', 'motorbike', 'mountain_biker', 'quad']
Initializing VehicleClassifier...
Setting up model: timm/eva02_large_patch14_448.mim_m38m_ft_in22k_in1k
Using timm model architecture
Freezing timm model layers...
Unfroze classifier head and last 3 blocks out of 24
Trainable parameters: 37,804,028 / 304,059,332 (12.4%)
Model loaded successfully


In [12]:
from pytorch_lightning.utilities.model_summary import ModelSummary

model_summary = ModelSummary(model)
print(model_summary)

  | Name  | Type | Params | Mode
--------------------------------------
0 | model | Eva  | 304 M  | eval
--------------------------------------
37.8 M    Trainable params
266 M     Non-trainable params
304 M     Total params
1,216.237 Total estimated model params size (MB)
0         Modules in train mode
564       Modules in eval mode


In [ ]:
image_size = model.hparams.get('image_size')

In [17]:
print(model.hparams)

"class_to_idx":  {'car_or_truck': 0, 'motorbike': 1, 'mountain_biker': 2, 'quad': 3}
"freeze_layers": True
"learning_rate": 0.0001
"model_name":    timm/eva02_large_patch14_448.mim_m38m_ft_in22k_in1k
"num_classes":   4


In [20]:
# Get transforms

transform = get_transforms_from_checkpoint(checkpoint_path, input_size=None)

Using image size from checkpoint: (448, 448)
Using normalization from checkpoint: mean (0.48145466, 0.4578275, 0.40821073), std (0.26862954, 0.26130258, 0.27577711)
Created transform pipeline: Compose(
    Resize(size=(448, 448), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)


In [ ]:
compiled_path = './exported-model/camera-trap-vehicle-classifier_compiled_cpu.pt2'
# script = model.to_torchscript()
# torch.jit.save(script, compiled_path)

from torch.export import export

# batch size of 1, 3 channels, image_size height and width
example_input = torch.randn(1, 3, 448, 448)

# export the model
exported_program = export(model, (example_input,))

# save for use in production environment
torch.export.save(exported_program, compiled_path)